In [ ]:
import streamlit as st
import pandas as pd
import pickle
import os

# FILE PATH
file_path = "churn_pipeline.pkl"

# SAFE LOAD PIPELINE
@st.cache_resource
def load_pipeline():
    global pipeline
    pipeline = pickle.load(open("churn_pipeline.pkl", "rb"))

    # Check file exists
    if not os.path.exists(file_path):
        st.error("❌ Model file not found! Please train and save the model first.")
        st.stop()

    # Check file not empty
    if os.path.getsize(file_path) == 0:
        st.error("❌ Model file is empty! Re-save your trained model.")
        st.stop()

    # Load safely
    try:
        with open(file_path, "rb") as f:
            pipeline = pickle.load(f)
    except Exception as e:
        st.error(f"❌ Error loading model: {e}")
        st.stop()

    return pipeline


def predict_customer(data):
    model = pipeline["model"]
    scaler = pipeline["scaler"]
    features = pipeline["features"]

    df_input = pd.DataFrame([data])
    
    df_input = df_input[features]
    df_scaled = scaler.transform(df_input)

    prediction = model.predict(df_scaled)
    return prediction[0]

    # Ensure all columns exist
    for col in features:
        if col not in df_input:
            df_input[col] = 0

    df_input = df_input[features]

    df_scaled = scaler.transform(df_input)

    pred = model.predict(df_scaled)[0]
    prob = model.predict_proba(df_scaled)[0][1]

    return pred, prob


# UI DESIGN
st.set_page_config(page_title="Customer Churn Prediction", layout="centered")

st.title("📊 Customer Churn Prediction System")
st.write("Predict whether a customer will churn or not")

# Debug info
with st.expander("🔍 Debug Info"):
    st.write("Current directory:", os.getcwd())
    st.write("Files:", os.listdir())


# USER INPUTS
tenure = st.slider("Tenure (months)", 0, 72, 12)
monthly_charges = st.number_input("Monthly Charges", 0.0, 10000.0, 100.0)
total_charges = st.number_input("Total Charges", 0.0, 100000.0, 1000.0)

phone_service = st.selectbox("Phone Service", ["Yes", "No"])
internet_service = st.selectbox("Internet Service", ["DSL", "Fiber optic", "No"])

contract = st.selectbox("Contract Type", ["Month-to-month", "One year", "Two year"])
payment_method = st.selectbox(
    "Payment Method",
    ["Electronic check", "Mailed check",
     "Bank transfer (automatic)", "Credit card (automatic)"]
)

if st.button("Predict"):

    data = {col: 0 for col in features}

    # Assign values
    data["tenure"] = tenure
    data["MonthlyCharges"] = monthly
    data["TotalCharges"] = total

    data["avg_monthly_spend"] = total / (tenure + 1)

    data[f"Contract_{contract}"] = 1
    data[f"InternetService_{internet}"] = 1
    data[f"PaymentMethod_{payment}"] = 1

    # ✅ MUST come before using prob
    pred, prob = predict_customer(data)

    st.subheader("Result")

    # Prediction result
    if pred == 1:
        st.error(f"⚠️ Customer will churn (Probability: {prob:.2f})")
    else:
        st.success(f"✅ Customer will stay (Probability: {prob:.2f})")

    # ✅ Now prob is defined → safe to use
    if prob > 0.7:
        st.error("High Risk 🔴")
    elif prob > 0.4:
        st.warning("Medium Risk 🟠")
    else:
        st.success("Low Risk 🟢")